In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.metrics import roc_curve, auc , confusion_matrix, accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split


from keras.models import load_model
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.optimizers import RMSprop


from dataset_preparation import awgn, LoadDataset, ChannelIndSpectrogram
from deep_learning_models import TripletNet, identity_loss

2024-05-24 18:41:49.212698: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-05-24 18:41:49.216472: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-05-24 18:41:49.261652: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-05-24 18:41:52.457592: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:



#!unzip -q /workspaces/work/dataset_training_aug.zip
training_dataset_path = "/workspaces/work/Dataset/dataset_training_aug.h5"
#"/workspaces/work/Dataset/dataset_training_no_aug.h5"
#"/workspaces/work/Dataset/dataset_training_aug.h5"


In [3]:
file_path = training_dataset_path        
dev_range = np.arange(0,30, dtype = int), 
pkt_range = np.arange(0,30000, dtype = int), 
#pkt_range = np.arange(0,1000, dtype = int),
snr_range = np.arange(20,80)

LoadDatasetObj = LoadDataset()
    
# Load preamble IQ samples and labels.
data, label = LoadDatasetObj.load_iq_samples(file_path, 
                                                 dev_range, 
                                                 pkt_range)

Dataset information: Dev 1 to Dev 30, 1000 packets per device.


In [6]:
data[0].shape

(8192,)

In [9]:
def train_feature_extractor(
    file_path = 
   # '/workspaces/work/Dataset/dataset_training_no_aug.h5'
    '/workspaces/work/Dataset/dataset_training_aug.h5'
    
        # './dataset/Train/dataset_training_aug.h5'  
        , 
    dev_range = np.arange(0,30, dtype = int), 
    pkt_range = np.arange(0,1000, dtype = int), 
    snr_range = np.arange(20,80)
                            ):

                            
    '''
    train_feature_extractor trains an RFF extractor using triplet loss.
    
    INPUT: 
        FILE_PATH is the path of training dataset.
        
        DEV_RANGE is the label range of LoRa devices to train the RFF extractor.
        
        PKT_RANGE is the range of packets from each LoRa device to train the RFF extractor.
        
        SNR_RANGE is the SNR range used in data augmentation. 
        
    RETURN:
        FEATURE_EXTRACTOR is the RFF extractor which can extract features from
        channel-independent spectrograms.
    '''
    
    LoadDatasetObj = LoadDataset()
    
    # Load preamble IQ samples and labels.
    data, label = LoadDatasetObj.load_iq_samples(file_path, 
                                                 dev_range, 
                                                 pkt_range)
        
    # Add additive Gaussian noise to the IQ samples.
    data = awgn(data, snr_range)
    
    ChannelIndSpectrogramObj = ChannelIndSpectrogram()
    
    # Convert time-domain IQ samples to channel-independent spectrograms.
    data = ChannelIndSpectrogramObj.channel_ind_spectrogram(data)
    
    return data, label
    

In [10]:
label.shape

(30000, 1)

In [9]:
label

array([[0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
    

In [11]:
np.unique(label)

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29])

In [12]:
data, label = train_feature_extractor()

# # VISUALISE A SAMPLE
test_sample = data[3].reshape(102,62)
plt.figure(figsize=(6, 3.2))
plt.imshow(test_sample, cmap='viridis', interpolation='nearest')
plt.colorbar()
plt.title('ColorMap')
plt.show()

Dataset information: Dev 1 to Dev 30, 1000 packets per device.


: 

In [ ]:
data[0].shape

In [ ]:
label[0].shape

In [ ]:
np.unique(label)

In [ ]:
#total_samples, height, width, _ = data.shape
#data_reshaped = data.reshape(total_samples, height, width)
#data_reshaped.shape

#data_real_values_train2, data_real_values_test2, label_train2, label_test2 = train_test_split(data_real_values, 
 #                                                                    label, 
  #                                                                  test_size=0.2, 
 #                                                                    shuffle= True)


#SOME PREPROCESSING



#data_real_values_train, data_real_values_test, label_train, label_test = train_test_split(data_real_values, label, test_size=0.2, random_state=30)



In [ ]:
total_samples, height, width, _ = data.shape
data_reshaped = data.reshape(total_samples, height, width)
data_reshaped.shape

data_reshaped_train2, data_reshaped_test2, label_train2, label_test2 = train_test_split(data_reshaped, 
                                                                     label, 
                                                                    test_size=0.2, 
                                                                    shuffle= True)

#SOME PREPROCESSING



#data_real_values_train, data_real_values_test, label_train, label_test = train_test_split(data_real_values, label, test_size=0.2, random_state=30)



In [ ]:
data_reshaped.shape

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

# Example data, replace with your actual data
data_reshaped = np.random.rand(15000, 102, 62)
labels = np.random.randint(0, 30, 15000)  # Example labels for 30 classes

# Reshape data to include the channel dimension, assuming grayscale images
data_reshaped = data_reshaped.reshape((15000, 102, 62, 1))

# Split the data into training and test sets
#data_train, X_test, y_train, y_test = train_test_split(data_reshaped, labels, test_size=0.2, random_state=42)

data_reshaped_train2, data_reshaped_test2, label_train2, label_test2 = train_test_split(data_reshaped, label, test_size=0.2)
print("Training set shape:", data_reshaped_train2.shape, label_train2.shape)
print("Test set shape:", data_reshaped_test2.shape, label_test2.shape)


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Data augmentation and normalization for training
train_datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    fill_mode='nearest'
)

# Normalization for test data (no augmentation)
test_datagen = ImageDataGenerator(rescale=1.0/255.0)

# Create data generators
train_generator = train_datagen.flow(data_reshaped_train2, label_train2, batch_size=32)
test_generator = test_datagen.flow(data_reshaped_test2, label_test2, batch_size=32)


In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, GlobalAveragePooling2D

def create_model_1(input_shape):
    input_layer = Input(shape=input_shape)
    x = Conv2D(32, (3, 3), activation='relu')(input_layer)
    x = MaxPooling2D((2, 2))(x)
    x = Flatten()(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x)
    output_layer = Dense(30, activation='softmax')(x)
    model = Model(inputs=input_layer, outputs=output_layer)
    return model

def create_model_2(input_shape):
    input_layer = Input(shape=input_shape)
    x = Conv2D(32, (3, 3), activation='relu')(input_layer)
    x = Conv2D(64, (3, 3), activation='relu')(x)
    x = MaxPooling2D((2, 2))(x)
    x = Flatten()(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x)
    output_layer = Dense(30, activation='softmax')(x)
    model = Model(inputs=input_layer, outputs=output_layer)
    return model

def create_model_3(input_shape):
    input_layer = Input(shape=input_shape)
    x = Conv2D(32, (3, 3), activation='relu')(input_layer)
    x = Conv2D(64, (3, 3), activation='relu')(x)
    x = Conv2D(128, (3, 3), activation='relu')(x)
    x = MaxPooling2D((2, 2))(x)
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.5)(x)
    output_layer = Dense(30, activation='softmax')(x)
    model = Model(inputs=input_layer, outputs=output_layer)
    return model

input_shape = (102, 62, 1)
model_1 = create_model_1(input_shape)
model_2 = create_model_2(input_shape)
model_3 = create_model_3(input_shape)

models = [model_1, model_2, model_3]


In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

callbacks = [
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=0.0001),
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
]

for model in models:
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    model.fit(
        train_generator,
        #steps_per_epoch=len(data_reshaped_train2) // 32,
        epochs=5, batch_size=32, # Adjust the number of epochs based on your needs
        validation_data=test_generator,
        #validation_steps=len(data_reshaped_test2) // 32,
        callbacks=callbacks,
    )


In [ ]:
import numpy as np

def ensemble_predictions(models, generator, steps):
    predictions = np.zeros((steps * generator.batch_size, 30))
    for model in models:
        predictions += model.predict(generator, steps=steps)
    return np.argmax(predictions, axis=1)

# Predict on test data
steps = len(X_test) // 32
ensemble_preds = ensemble_predictions(models, test_generator, steps=steps)


In [ ]:
from sklearn.metrics import accuracy_score

y_true = y_test[:len(ensemble_preds)]  # Ensure the length matches
accuracy = accuracy_score(y_true, ensemble_preds)
print(f'Ensemble test accuracy: {accuracy * 100:.2f}%')


In [ ]:
finish

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

# Example data, replace with your actual data
data_reshaped = np.random.rand(15000, 102, 62)
labels = np.random.randint(0, 30, 15000)  # Example labels for 30 classes

# Reshape data to include the channel dimension, assuming grayscale images
data_reshaped = data_reshaped.reshape((15000, 102, 62, 1))

# One-hot encode the labels
#labels = to_categorical(labels, num_classes=30)

data_reshaped_train2, data_reshaped_test2, label_train2, label_test2 = train_test_split(data_reshaped, 
                                                                     label, 
                                                                    test_size=0.2, 
                                                                    random_state=42)

  
  
print("Training set shape:", data_reshaped_train2.shape, label_train2.shape)
print("Test set shape:", data_reshaped_test2.shape, label_test2.shape)                                                                  

# Split the data into training and test sets
#X_train, X_test, y_train, y_test = train_test_split(data_reshaped, labels, test_size=0.2, random_state=42)

#print("Training set shape:", X_train.shape, y_train.shape)
#print("Test set shape:", X_test.shape, y_test.shape)


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Data augmentation and normalization for training
train_datagen = ImageDataGenerator(
    rescale=1.0/255.0,            # Normalize the pixel values to [0, 1]
    shear_range=0.2,              # Apply shear transformation
    zoom_range=0.2,               # Apply zoom transformation
    horizontal_flip=True,         # Randomly flip images horizontally
    rotation_range=20,            # Randomly rotate images by up to 20 degrees
    width_shift_range=0.2,        # Randomly shift images horizontally by 20% of the width
    height_shift_range=0.2,       # Randomly shift images vertically by 20% of the height
    fill_mode='nearest'           # Fill mode for points outside the boundaries
)

# Normalization for test data (no augmentation)
test_datagen = ImageDataGenerator(rescale=1.0/255.0)

# Create data generators
train_generator = train_datagen.flow(data_reshaped_train2, label_train2, batch_size=32)
test_generator = test_datagen.flow(data_reshaped_test2, label_test2, batch_size=32)


In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, GlobalAveragePooling2D

# Adjust input to 3 channels
input_tensor = Input(shape=(102, 62, 3))
x = Conv2D(3, (1, 1), padding='same')(input_tensor)

# Load VGG16 model, pre-trained on ImageNet, without top layers
base_model = VGG16(weights='imagenet', include_top=False, input_tensor=x)

# Add custom top layers
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(30, activation='softmax')(x)

# Define the model
model = Model(inputs=base_model.input, outputs=predictions)

# Freeze the layers of VGG16 base model
for layer in base_model.layers:
    layer.trainable = False

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Print the model summary
#model.summary()

reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=0.0001)
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
# Train the model
history = model.fit(
    train_generator,
    #steps_per_epoch=len(data_reshaped_train2) // 32,
    #epochs=50,  # Adjust the number of epochs based on your needs
    validation_data=test_generator,
    epochs=1000, batch_size=32,
    #validation_steps=len(data_reshaped_test2) // 32
    callbacks=[reduce_lr, early_stopping],
    
)

# Evaluate the model on the test set
test_loss, test_acc = model.evaluate(test_generator)
print(f'Test accuracy: {test_acc * 100:.2f}%')

# Save the model architecture and weights
model.save('my_model.h5')

# Load the model architecture and weights
from tensorflow.keras.models import load_model

loaded_model = load_model('my_model.h5')

# Check the loaded model summary to confirm it matches the saved model
loaded_model.summary()

# Evaluate the loaded model to ensure everything is working correctly
test_loss, test_acc = loaded_model.evaluate(test_generator)
print(f'Test accuracy of loaded model: {test_acc * 100:.2f}%')

In [ ]:
#with hyperfine tuning 
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
    

#loss1 = 'sparse_categorical_crossentropy'
#loss2='categorical_crossentropy'
#loss3 = keras.losses.BinaryFocalCrossentropy()


#optimizer = keras.optimizers.Adam(learning_rate=0.001) #https://keras.io/api/optimizers/adam/
#optimizer = keras.optimizers.RMSprop(learning_rate=0.001) # https://keras.io/api/optimizers/rmsprop/
#optimizer = keras.optimizers.SGD(learning_rate=0.001) #https://keras.io/api/optimizers/sgd/


#cnn_model2.compile(optimizer=optimizer, loss=loss2, metrics=['accuracy'])
#train_history =model.fit(data_reshaped_train2, label_train2, 
 #                               validation_data = (data_reshaped_test2, label_test2),
  #                                 epochs=50, batch_size=100)

reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=0.0001)
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
# Train the model
history = model.fit(
    train_generator,
    #steps_per_epoch=len(data_reshaped_train2) // 32,
    #epochs=50,  # Adjust the number of epochs based on your needs
    validation_data=test_generator,
    epochs=1000, batch_size=32
    #validation_steps=len(data_reshaped_test2) // 32
    callbacks=[reduce_lr, early_stopping]
    
)

# Evaluate the model on the test set
test_loss, test_acc = model.evaluate(test_generator)
print(f'Test accuracy: {test_acc * 100:.2f}%')

In [ ]:
#with  hyperfine tuning
import tensorflow as tf
import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import  Conv1D, Conv2D, MaxPooling1D, MaxPooling2D, Flatten, Dense, BatchNormalization
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, GRU, SimpleRNN


cnn_model2 = Sequential()
cnn_model2.add(Conv2D(filters=32, kernel_size=(3,3), activation='relu', padding='same', input_shape=(height, width, 1)))
cnn_model2.add(BatchNormalization())
cnn_model2.add(MaxPooling2D(pool_size=(2, 2)))
#cnn_model2.add(MaxPooling1D(pool_size=2, padding='same'))
cnn_model2.add(Dropout(0.25))

cnn_model2.add(Conv2D(64, (3, 3), activation='relu'))
cnn_model2.add(BatchNormalization())
cnn_model2.add(MaxPooling2D(pool_size=(2, 2)))
cnn_model2.add(Dropout(0.25))

cnn_model2.add(Conv2D(64, (3, 3), activation='relu'))
cnn_model2.add(BatchNormalization())
cnn_model2.add(MaxPooling2D(pool_size=(2, 2)))
cnn_model2.add(Dropout(0.25))

cnn_model2.add(Conv2D(64, (3, 3), activation='relu'))
cnn_model2.add(BatchNormalization())
cnn_model2.add(MaxPooling2D(pool_size=(2, 2)))
cnn_model2.add(Dropout(0.25))

cnn_model2.add(Flatten())

cnn_model2.add(Dense(512, activation='relu'))
cnn_model2.add(Dropout(0.5))
cnn_model2.add(Dense(256, activation='relu'))
cnn_model2.add(Dropout(0.5))
#.add(Conv2D(64, kernel_size=(3,3), activation='relu'))
#cnn_model2.add(Conv2D(128, kernel_size=1, activation='relu'))
#cnn_model2.add(Flatten())
#cnn_model2.add(Dense(64, activation='relu'))
cnn_model2.add(Dense(30, activation='softmax'))







#cnn_model2.summary()



loss1 = 'sparse_categorical_crossentropy'
#loss2='categorical_crossentropy'
#loss3 = keras.losses.BinaryFocalCrossentropy()


optimizer = keras.optimizers.Adam(learning_rate=0.001) #https://keras.io/api/optimizers/adam/
#optimizer = keras.optimizers.RMSprop(learning_rate=0.001) # https://keras.io/api/optimizers/rmsprop/
#optimizer = keras.optimizers.SGD(learning_rate=0.001) #https://keras.io/api/optimizers/sgd/


cnn_model2.compile(optimizer=optimizer, loss=loss1, metrics=['accuracy'])
#train_history =cnn_model2.fit(data_reshaped_train2, label_train2, 
 #                               validation_data = (data_reshaped_test2, label_test2),
  #                                 epochs=50, batch_size=100)

reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=0.0001)
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
# Train the model
history = cnn_model2.fit(
    train_generator,
    #steps_per_epoch=len(data_reshaped_train2) // 32,
    #epochs=50,  # Adjust the number of epochs based on your needs
    validation_data=test_generator,
    epochs=10, batch_size=32,
    #validation_steps=len(data_reshaped_test2) // 32
    callbacks=[reduce_lr, early_stopping],
    
)

# Evaluate the model on the test set
test_loss, test_acc = model.evaluate(test_generator)
print(f'Test accuracy: {test_acc * 100:.2f}%')



In [ ]:
import tensorflow as tf
import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import  Conv1D, Conv2D, MaxPooling1D, MaxPooling2D, Flatten, Dense, BatchNormalization
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, GRU, SimpleRNN


cnn_model2 = Sequential()
cnn_model2.add(Conv2D(filters=32, kernel_size=(3,3), activation='relu', padding='same', input_shape=(height, width, 1)))
cnn_model2.add(BatchNormalization())
cnn_model2.add(MaxPooling2D(pool_size=(2, 2)))
#cnn_model2.add(MaxPooling1D(pool_size=2, padding='same'))
cnn_model2.add(Dropout(0.25))

cnn_model2.add(Conv2D(64, (3, 3), activation='relu'))
cnn_model2.add(BatchNormalization())
cnn_model2.add(MaxPooling2D(pool_size=(2, 2)))
cnn_model2.add(Dropout(0.25))

cnn_model2.add(Conv2D(64, (3, 3), activation='relu'))
cnn_model2.add(BatchNormalization())
cnn_model2.add(MaxPooling2D(pool_size=(2, 2)))
cnn_model2.add(Dropout(0.25))

cnn_model2.add(Conv2D(64, (3, 3), activation='relu'))
cnn_model2.add(BatchNormalization())
cnn_model2.add(MaxPooling2D(pool_size=(2, 2)))
cnn_model2.add(Dropout(0.25))

cnn_model2.add(Flatten())

cnn_model2.add(Dense(512, activation='relu'))
cnn_model2.add(Dropout(0.5))
cnn_model2.add(Dense(256, activation='relu'))
cnn_model2.add(Dropout(0.5))
#.add(Conv2D(64, kernel_size=(3,3), activation='relu'))
#cnn_model2.add(Conv2D(128, kernel_size=1, activation='relu'))
#cnn_model2.add(Flatten())
#cnn_model2.add(Dense(64, activation='relu'))
cnn_model2.add(Dense(30, activation='softmax'))







#cnn_model2.summary()



loss1 = 'sparse_categorical_crossentropy'
#loss2='categorical_crossentropy'
#loss3 = keras.losses.BinaryFocalCrossentropy()


optimizer = keras.optimizers.Adam(learning_rate=0.001) #https://keras.io/api/optimizers/adam/
#optimizer = keras.optimizers.RMSprop(learning_rate=0.001) # https://keras.io/api/optimizers/rmsprop/
#optimizer = keras.optimizers.SGD(learning_rate=0.001) #https://keras.io/api/optimizers/sgd/


cnn_model2.compile(optimizer=optimizer, loss=loss1, metrics=['accuracy'])
#train_history =cnn_model2.fit(data_reshaped_train2, label_train2, 
 #                               validation_data = (data_reshaped_test2, label_test2),
  #                                 epochs=50, batch_size=100)

reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=0.0001)
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
# Train the model
history = cnn_model2.fit(
    train_generator,
    #steps_per_epoch=len(data_reshaped_train2) // 32,
    #epochs=50,  # Adjust the number of epochs based on your needs
    validation_data=test_generator,
    epochs=10, batch_size=32,
    #validation_steps=len(data_reshaped_test2) // 32
    callbacks=[reduce_lr, early_stopping],
   
    
)

# Evaluate the model on the test set
test_loss, test_acc = model.evaluate(test_generator)
print(f'Test accuracy: {test_acc * 100:.2f}%')



In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization

# Initialize the model
model = Sequential()

# Add convolutional layers with batch normalization and dropout
model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(102, 62, 1)))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

model.add(Conv2D(256, (3, 3), activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

model.add(Flatten())

model.add(Dense(512, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(256, activation='relu'))
model.add(Dropout(0.5))

# Output layer with the number of classes (e.g., 30 for 30 classes)
model.add(Dense(30, activation='softmax'))

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Print the model summary
#model.summary()


In [ ]:
# Train the model
history = model.fit(
    train_generator,
    steps_per_epoch=len(data_reshaped_train2) // 32,
    epochs=50,  # Adjust the number of epochs based on your needs
    validation_data=test_generator,
    validation_steps=len(data_reshaped_test2) // 32
)

# Evaluate the model on the test set
test_loss, test_acc = model.evaluate(test_generator)
print(f'Test accuracy: {test_acc * 100:.2f}%')


In [ ]:
import tensorflow as tf
import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import  Conv1D, Conv2D, MaxPooling1D, MaxPooling2D, Flatten, Dense, BatchNormalization
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, GRU, SimpleRNN


cnn_model2 = Sequential()
cnn_model2.add(Conv2D(filters=32, kernel_size=(3,3), activation='relu', padding='same', input_shape=(height, width, 1)))
cnn_model2.add(BatchNormalization())
cnn_model2.add(MaxPooling2D(pool_size=(2, 2)))
#cnn_model2.add(MaxPooling1D(pool_size=2, padding='same'))
cnn_model2.add(Dropout(0.25))

cnn_model2.add(Conv2D(64, (3, 3), activation='relu'))
cnn_model2.add(BatchNormalization())
cnn_model2.add(MaxPooling2D(pool_size=(2, 2)))
cnn_model2.add(Dropout(0.25))

cnn_model2.add(Conv2D(64, (3, 3), activation='relu'))
cnn_model2.add(BatchNormalization())
cnn_model2.add(MaxPooling2D(pool_size=(2, 2)))
cnn_model2.add(Dropout(0.25))

cnn_model2.add(Conv2D(64, (3, 3), activation='relu'))
cnn_model2.add(BatchNormalization())
cnn_model2.add(MaxPooling2D(pool_size=(2, 2)))
cnn_model2.add(Dropout(0.25))

cnn_model2.add(Flatten())

cnn_model2.add(Dense(512, activation='relu'))
cnn_model2.add(Dropout(0.5))
cnn_model2.add(Dense(256, activation='relu'))
cnn_model2.add(Dropout(0.5))
#.add(Conv2D(64, kernel_size=(3,3), activation='relu'))
#cnn_model2.add(Conv2D(128, kernel_size=1, activation='relu'))
#cnn_model2.add(Flatten())
#cnn_model2.add(Dense(64, activation='relu'))
cnn_model2.add(Dense(30, activation='softmax'))







#cnn_model2.summary()



loss1 = 'sparse_categorical_crossentropy'
#loss2 = 'binary_crossentropy'
#loss3 = keras.losses.BinaryFocalCrossentropy()


optimizer = keras.optimizers.Adam(learning_rate=0.1) #https://keras.io/api/optimizers/adam/
#optimizer = keras.optimizers.RMSprop(learning_rate=0.001) # https://keras.io/api/optimizers/rmsprop/
#optimizer = keras.optimizers.SGD(learning_rate=0.001) #https://keras.io/api/optimizers/sgd/


#X_train, y_train,X_val, y_val,
cnn_model2.compile(optimizer=optimizer, loss=loss1, metrics=['accuracy'])
train_history =cnn_model2.fit(X_train, y_train, 
                                   validation_data = (X_val,y_val),
                                   epochs=50, batch_size=32)

#cnn_model2.compile(optimizer=optimizer, loss=loss1, metrics=['accuracy'])
#train_history =cnn_model2.fit(data_reshaped_train2, label_train2, 
 #                                  validation_data = (data_reshaped_test2, label_test2),
 #                                  epochs=50, batch_size=100)




In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization

# Initialize the model
model = Sequential()

# Add convolutional layers with batch normalization and dropout
model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(102, 62, 1)))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

model.add(Conv2D(256, (3, 3), activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

model.add(Flatten())

model.add(Dense(512, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(256, activation='relu'))
model.add(Dropout(0.5))

# Output layer with the number of classes (e.g., 30 for 30 classes)
model.add(Dense(30, activation='softmax'))

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Print the model summary
#model.summary()


In [ ]:
import tensorflow as tf
import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import  Conv1D, Conv2D, MaxPooling1D, MaxPooling2D, Flatten, Dense, BatchNormalization
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, GRU, SimpleRNN


cnn_model2 = Sequential()
cnn_model2.add(Conv2D(filters=32, kernel_size=(3,3), activation='relu', padding='same', input_shape=(height, width, 1)))
cnn_model2.add(BatchNormalization())
cnn_model2.add(MaxPooling2D(pool_size=(2, 2)))
#cnn_model2.add(MaxPooling1D(pool_size=2, padding='same'))
cnn_model2.add(Dropout(0.25))

cnn_model2.add(Conv2D(64, (3, 3), activation='relu'))
cnn_model2.add(BatchNormalization())
cnn_model2.add(MaxPooling2D(pool_size=(2, 2)))
cnn_model2.add(Dropout(0.25))

cnn_model2.add(Conv2D(64, (3, 3), activation='relu'))
cnn_model2.add(BatchNormalization())
cnn_model2.add(MaxPooling2D(pool_size=(2, 2)))
cnn_model2.add(Dropout(0.25))

cnn_model2.add(Conv2D(64, (3, 3), activation='relu'))
cnn_model2.add(BatchNormalization())
cnn_model2.add(MaxPooling2D(pool_size=(2, 2)))
cnn_model2.add(Dropout(0.25))

cnn_model2.add(Flatten())

cnn_model2.add(Dense(512, activation='relu'))
cnn_model2.add(Dropout(0.5))
cnn_model2.add(Dense(256, activation='relu'))
cnn_model2.add(Dropout(0.5))
#.add(Conv2D(64, kernel_size=(3,3), activation='relu'))
#cnn_model2.add(Conv2D(128, kernel_size=1, activation='relu'))
#cnn_model2.add(Flatten())
#cnn_model2.add(Dense(64, activation='relu'))
cnn_model2.add(Dense(30, activation='softmax'))







#cnn_model2.summary()



loss1 = 'sparse_categorical_crossentropy'
#loss2 = 'binary_crossentropy'
#loss3 = keras.losses.BinaryFocalCrossentropy()


optimizer = keras.optimizers.Adam(learning_rate=0.1) #https://keras.io/api/optimizers/adam/
#optimizer = keras.optimizers.RMSprop(learning_rate=0.001) # https://keras.io/api/optimizers/rmsprop/
#optimizer = keras.optimizers.SGD(learning_rate=0.001) #https://keras.io/api/optimizers/sgd/




cnn_model2.compile(optimizer=optimizer, loss=loss1, metrics=['accuracy'])
train_history =cnn_model2.fit(data_reshaped_train2, label_train2, 
                                   validation_data = (data_reshaped_test2, label_test2),
                                   epochs=50, batch_size=32)




In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# Define the input shape of your images (e.g., 64x64 pixels, 3 color channels)
input_shape = (64, 64, 3)

# Initialize the model
cnn_model3 = Sequential()

# Add convolutional layers, pooling layers, and fully connected layers
cnn_model3.add(Conv2D(32, (3, 3), activation='relu', input_shape=input_shape))
cnn_model3.add(MaxPooling2D(pool_size=(2, 2)))

cnn_model3.add(Conv2D(64, (3, 3), activation='relu'))
cnn_model3.add(MaxPooling2D(pool_size=(2, 2)))

cnn_model3.add(Conv2D(128, (3, 3), activation='relu'))
cnn_model3.add(MaxPooling2D(pool_size=(2, 2)))

cnn_model3.add(Flatten())

cnn_model3.add(Dense(128, activation='relu'))
cnn_model3.add(Dropout(0.5))  # Dropout for regularization
cnn_model3.add(Dense(64, activation='relu'))

# Output layer with 31 units for 31 classes (0 to 30) and softmax activation
cnn_model3.add(Dense(30, activation='softmax'))

# Compile the model
cnn_model3.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

train_history = cnn_model3.fit(data_reshaped_train2, label_train2, 
                                   validation_data = (data_reshaped_test2, label_test2),
                                   epochs=1, batch_size=32)



# Print the model summary
#model.summary()


In [ ]:
# Create a Sequential model
mlp_model = Sequential()

# Add layers to the model
mlp_model.add(Dense(128, activation='relu', input_shape=(height, width, 1)))  # Input layer with 10 input features and 128 neurons
mlp_model.add(Dense(64, activation='relu'))                # Hidden layer with 64 neurons and ReLU activation
mlp_model.add(Dense(32, activation='relu'))                # Another hidden layer with 32 neurons and ReLU activation
mlp_model.add(Dense(30, activation='sigmoid'))              # Output layer with 1 neuron and sigmoid activation for binary classification



# Print the model summary
#model.summary()

In [ ]:
# Define the LSTM model
lstm_model = Sequential()
lstm_model.add(LSTM(128, activation='relu', input_shape=(height, width)))
lstm_model.add(Dense(128, activation='relu'))
lstm_model.add(Dropout(0.5))
lstm_model.add(Dense(1, activation='sigmoid'))


In [ ]:
simpleRNN_model = Sequential()

# Add SimpleRNN layer to the model
simpleRNN_model.add(SimpleRNN(64, input_shape=(height, width)))  # SimpleRNN layer with 64 units and input shape (10, 1)

# Add Dense layer to the model
simpleRNN_model.add(Dense(1, activation='sigmoid'))  # Output layer with sigmoid activation for binary classification
#model.summary()

In [ ]:
gru_model = Sequential()

# Add GRU layer to the model
gru_model.add(GRU(64, input_shape=(height, width)))  # GRU layer with 64 units and input shape (10, 1)

# Add Dense layer to the model
gru_model.add(Dense(1, activation='sigmoid'))  # Output layer with sigmoid activation for binary classification

In [ ]:
#for CNN_LSTM
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, LSTM, Dense, SimpleRNN, GRU

# Create a Sequential model for CNN
cnn_model = Sequential()
cnn_model.add(Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(102, 62, 1)))
cnn_model.add(MaxPooling2D(pool_size=(2, 2)))
cnn_model.add(Conv2D(64, kernel_size=(3, 3), activation='relu'))
cnn_model.add(MaxPooling2D(pool_size=(2, 2)))
cnn_model.add(Flatten())
cnn_model.add(Dense(128, activation='relu'))
cnn_model.add(Dense(1, activation='sigmoid'))

# Create a Sequential model for LSTM
lstm_model = Sequential()
lstm_model.add(LSTM(128, input_shape=(102, 62)))
lstm_model.add(Dense(1, activation='sigmoid'))

#Compile models
models = [cnn_model, lstm_model, mlp_model]
for model in models:
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

train_history = model.fit(data_reshaped_train2, label_train2, 
                                   validation_data = (data_reshaped_test2, label_test2),
                                   epochs=1, batch_size=32)

In [ ]:
#for CNN_LSTM_MLP
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, LSTM, Dense, SimpleRNN, GRU

# Create a Sequential model for CNN
cnn_model = Sequential()
cnn_model.add(Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(102, 62, 1)))
cnn_model.add(MaxPooling2D(pool_size=(2, 2)))
cnn_model.add(Conv2D(64, kernel_size=(3, 3), activation='relu'))
cnn_model.add(MaxPooling2D(pool_size=(2, 2)))
cnn_model.add(Flatten())
cnn_model.add(Dense(128, activation='relu'))
cnn_model.add(Dense(1, activation='sigmoid'))

# Create a Sequential model for LSTM
lstm_model = Sequential()
lstm_model.add(LSTM(128, input_shape=(102, 62)))
lstm_model.add(Dense(1, activation='sigmoid'))

# Create a Sequential model for MLP
mlp_model = Sequential()
mlp_model.add(Dense(256, activation='relu', input_shape=(102, 62)))
mlp_model.add(Flatten())
mlp_model.add(Dense(128, activation='relu'))
mlp_model.add(Dense(1, activation='sigmoid'))

 #Compile models
models = [cnn_model, lstm_model, mlp_model]
for model in models:
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

train_history = model.fit(data_reshaped_train2, label_train2, 
                                   validation_data = (data_reshaped_test2, label_test2),
                                   epochs=1, batch_size=32)

In [ ]:
#for CNN_LSTM_MLP_SIMPLERNN
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, LSTM, Dense, SimpleRNN, GRU

# Create a Sequential model for CNN
cnn_model = Sequential()
cnn_model.add(Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(102, 62, 1)))
cnn_model.add(MaxPooling2D(pool_size=(2, 2)))
cnn_model.add(Conv2D(64, kernel_size=(3, 3), activation='relu'))
cnn_model.add(MaxPooling2D(pool_size=(2, 2)))
cnn_model.add(Flatten())
cnn_model.add(Dense(128, activation='relu'))
cnn_model.add(Dense(1, activation='sigmoid'))

# Create a Sequential model for LSTM
lstm_model = Sequential()
lstm_model.add(LSTM(128, input_shape=(102, 62)))
lstm_model.add(Dense(1, activation='sigmoid'))

# Create a Sequential model for MLP
mlp_model = Sequential()
mlp_model.add(Dense(256, activation='relu', input_shape=(102, 62)))
mlp_model.add(Flatten())
mlp_model.add(Dense(128, activation='relu'))
mlp_model.add(Dense(1, activation='sigmoid'))

# Create a Sequential model for SimpleRNN
simplernn_model = Sequential()
simplernn_model.add(SimpleRNN(128, input_shape=(102, 62)))
simplernn_model.add(Dense(1, activation='sigmoid'))

models = [cnn_model, lstm_model, mlp_model, simplernn_model, gru_model]
for model in models:
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    
train_history = model.fit(data_reshaped_train2, label_train2, 
                                   validation_data = (data_reshaped_test2, label_test2),
                                   epochs=1, batch_size=32)

In [ ]:
#for CNN_LSTM_MLP_SIMPLERNN_GRU
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, LSTM, Dense, SimpleRNN, GRU

# Create a Sequential model for CNN
cnn_model = Sequential()
cnn_model.add(Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(102, 62, 1)))
cnn_model.add(MaxPooling2D(pool_size=(2, 2)))
cnn_model.add(Conv2D(64, kernel_size=(3, 3), activation='relu'))
cnn_model.add(MaxPooling2D(pool_size=(2, 2)))
cnn_model.add(Flatten())
cnn_model.add(Dense(128, activation='relu'))
cnn_model.add(Dense(1, activation='sigmoid'))

# Create a Sequential model for LSTM
lstm_model = Sequential()
lstm_model.add(LSTM(128, input_shape=(102, 62)))
lstm_model.add(Dense(1, activation='sigmoid'))

# Create a Sequential model for MLP
mlp_model = Sequential()
mlp_model.add(Dense(256, activation='relu', input_shape=(102, 62)))
mlp_model.add(Flatten())
mlp_model.add(Dense(128, activation='relu'))
mlp_model.add(Dense(1, activation='sigmoid'))

# Create a Sequential model for SimpleRNN
simplernn_model = Sequential()
simplernn_model.add(SimpleRNN(128, input_shape=(102, 62)))
simplernn_model.add(Dense(1, activation='sigmoid'))

# Create a Sequential model for GRU
gru_model = Sequential()
gru_model.add(GRU(128, input_shape=(102, 62)))
gru_model.add(Dense(1, activation='sigmoid'))

# Compile models
models = [cnn_model, lstm_model, mlp_model, simplernn_model, gru_model]
for model in models:
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Print model summaries
#for i, model in enumerate(models):
  #  print(f"Model {i+1} Summary:")
  #  print(model.summary())
  #  print("\n")

train_history = model.fit(data_reshaped_train2, label_train2, 
                                   validation_data = (data_reshaped_test2, label_test2),
                                   epochs=1, batch_size=32)

In [ ]:
model_to_train = cnn_model2
#model_to_train = cnn_model
#model_to_train = lstm_model
#model_to_train = CuDNNlstm_model
#model_to_train = cnn_lstm_model
#model_to_train = mlp_model
#model_to_train = simpleRNN_model
#model_to_train = gru_model
#model_to_train=CNN_LSTM_model

loss1 = 'sparse_categorical_crossentropy'
#loss2 = 'binary_crossentropy'
#loss3 = keras.losses.BinaryFocalCrossentropy()


#optimizer = keras.optimizers.Adam(learning_rate=0.0001) #https://keras.io/api/optimizers/adam/
optimizer = keras.optimizers.RMSprop(learning_rate=0.001) # https://keras.io/api/optimizers/rmsprop/
#optimizer = keras.optimizers.SGD(learning_rate=0.001) #https://keras.io/api/optimizers/sgd/




model_to_train.compile(optimizer=optimizer, loss=loss1, metrics=['accuracy'])
train_history = model_to_train.fit(data_reshaped_train2, label_train2, 
                                   validation_data = (data_reshaped_test2, label_test2),
                                   epochs=100, batch_size=32)


#model_to_train.compile(optimizer=optimizer, loss=loss3, metrics=['accuracy'])

#train_history = model_to_train.fit(data, label, 
#                                   epochs=1, batch_size=32)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, LSTM, Dense, TimeDistributed, Flatten

# Define input shape for CNN
input_shape = (10, 64, 64, 3)  # 10 frames of 64x64 RGB images

# Create a Sequential model
model = Sequential()

# CNN layers
model.add(TimeDistributed(Conv2D(32, (3, 3), activation='relu'), input_shape=input_shape))
model.add(TimeDistributed(MaxPooling2D((2, 2))))
model.add(TimeDistributed(Conv2D(64, (3, 3), activation='relu')))
model.add(TimeDistributed(MaxPooling2D((2, 2))))
model.add(TimeDistributed(Flatten()))

# LSTM layer
model.add(LSTM(128, return_sequences=False))

# Output layer
model.add(Dense(1, activation='sigmoid'))

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Print the model summary
#model.summary()

In [ ]:
model_to_train.save('2_first_model.h5')

In [ ]:
# load and evaluate a saved model
from numpy import loadtxt
from tensorflow.keras.models import load_model
 
# load model
model = load_model('2_first_model.h5')
# summarize model.
model.summary()

# evaluate the model
score = model.evaluate(data_real_values_test, label_test, verbose=0)
print("%s: %.2f%%" % (model.metrics_names[1], score[1]*100))

In [ ]:
import pandas as pd
pd.DataFrame(train_history.history).plot(figsize=(8,5))
plt.title(f'hey')
# plt.savefig('./Plots/Base_model/7.')
plt.show()

In [ ]:
print(np.unique(label))

In [ ]:
# data_real_values_test, label_train, label_testa
all_labels = np.unique(label)
predictions = model_to_train.predict(data_real_values_test)
confusion_matrix = metrics.confusion_matrix(label_test, predictions)
precision_recall_fscore_support(label_test, predictions, average='macro')
cm_display = metrics.ConfusionMatrixDisplay(confusion_matrix = confusion_matrix, display_labels = all_labels)
cm_display.plot()
plt.show()